In [1]:
!pip install yfinance backtesting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 2.9 MB/s eta 0:00:00


In [9]:
import yfinance as yf
import pandas as pd
import numpy as np
from backtesting import Backtest, Strategy


In [3]:
data = yf.download("RELIANCE.NS", period="1y", interval="1h")

data.dropna(inplace=True)

if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)


/tmp/ipykernel_1347/1084645603.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download("RELIANCE.NS", period="1y", interval="1h")
[*********************100%***********************]  1 of 1 completed


In [4]:
def EMA(series, n):
    series = np.array(series, dtype=float)
    alpha  = 2 / (n + 1)
    ema    = np.empty(len(series))
    ema[0] = series[0]

    for i in range(1, len(series)):
        ema[i] = alpha * series[i] + (1 - alpha) * ema[i-1]

    return ema


In [5]:
def RSI(series, n=14):
    series   = np.array(series, dtype=float)
    delta    = np.diff(series, prepend=series[0])

    gain     = np.where(delta > 0, delta, 0.0)
    loss     = np.where(delta < 0, -delta, 0.0)

    avg_gain = np.empty(len(series))
    avg_loss = np.empty(len(series))

    avg_gain[n] = gain[1:n+1].mean()
    avg_loss[n] = loss[1:n+1].mean()

    for i in range(n+1, len(series)):
        avg_gain[i] = (avg_gain[i-1] * (n-1) + gain[i]) / n
        avg_loss[i] = (avg_loss[i-1] * (n-1) + loss[i]) / n

    rs         = np.zeros(len(series))
    rsi        = np.full(len(series), np.nan)
    rs[n:]     = avg_gain[n:] / np.where(avg_loss[n:] == 0, 1e-10, avg_loss[n:])
    rsi[n:]    = 100 - (100 / (1 + rs[n:]))

    return rsi

In [6]:
def ATR(df, n=14):
    high  = np.array(df['High'],  dtype=float)
    low   = np.array(df['Low'],   dtype=float)
    close = np.array(df['Close'], dtype=float)

    prev_close = np.roll(close, 1)
    prev_close[0] = close[0]

    tr  = np.maximum(high - low,
          np.maximum(np.abs(high - prev_close),
                     np.abs(low  - prev_close)))

    atr = np.full(len(tr), np.nan)
    atr[n] = tr[1:n+1].mean()

    for i in range(n+1, len(tr)):
        atr[i] = (atr[i-1] * (n-1) + tr[i]) / n

    return atr

In [7]:
class EMARSIATRStrategy(Strategy):

    rsi_threshold = 42
    rsi_window    = 7
    sl_multiplier = 2.0
    ema_lookback  = 10

    def init(self):
        close    = self.data.Close
        self.ema = self.I(EMA, close, 50)
        self.rsi = self.I(RSI, close, 14)
        self.atr = self.I(ATR, self.data.df, 14)

    def next(self):
        price = self.data.Close[-1]
        atr   = self.atr[-1]
        rsi   = self.rsi[-1]

        if np.isnan(self.ema[-1]) or np.isnan(rsi) or np.isnan(atr):
            return

        big_trend_up   = self.ema[-1] > self.ema[-self.ema_lookback]
        big_trend_down = self.ema[-1] < self.ema[-self.ema_lookback]

        rsi_recovered = any(
            self.rsi[-k-1] < self.rsi_threshold and self.rsi[-k]   > self.rsi_threshold
            for k in range(1, self.rsi_window + 1)
        )
        rsi_faded = any(
            self.rsi[-k-1] > (100 - self.rsi_threshold) and self.rsi[-k]   < (100 - self.rsi_threshold)
            for k in range(1, self.rsi_window + 1)
        )

        if big_trend_up and rsi_recovered and not self.position:
            sl = price - self.sl_multiplier * atr
            tp = price + 2 * (price - sl)
            self.buy(sl=sl, tp=tp)

        elif big_trend_down and rsi_faded and not self.position:
            sl = price + self.sl_multiplier * atr
            tp = price - 2 * (sl - price)
            self.sell(sl=sl, tp=tp)


bt_opt = Backtest(
    data,
    EMARSIATRStrategy,
    cash             = 100000,
    commission       = 0.0005,
    exclusive_orders = True,
    finalize_trades  = True
)

stats = bt_opt.run()

print(stats)



Backtest.run:   0%|          | 0/1695 [00:00<?, ?bar/s]

Start                     2025-04-01 03:45...
End                       2026-03-30 09:45...
Duration                    363 days 06:00:00
Exposure Time [%]                    22.74854
Equity Final [$]                 113416.38479
Equity Peak [$]                  116228.28547
Commissions [$]                    2503.85156
Return [%]                           13.41638
Buy & Hold Return [%]                 8.35976
Return (Ann.) [%]                    13.82508
Volatility (Ann.) [%]                 9.41527
CAGR [%]                               9.1266
Sharpe Ratio                          1.46837
Sortino Ratio                         3.16978
Calmar Ratio                          2.88755
Alpha [%]                            13.53738
Beta                                 -0.01447
Max. Drawdown [%]                    -4.78783
Avg. Drawdown [%]                    -1.10844
Max. Drawdown Duration      112 days 00:00:00
Avg. Drawdown Duration       12 days 01:00:00
# Trades                          

In [8]:
opt_stats = bt_opt.optimize(
    rsi_threshold  = [35, 38, 40, 42, 45],
    rsi_window     = [ 5, 7, 9],
    sl_multiplier  = [ 2.0, 2.5],
    ema_lookback   = [ 5,10,15],
    maximize       = 'Sharpe Ratio',
)

print(f"  rsi_threshold : {opt_stats._strategy.rsi_threshold}")
print(f"  rsi_window    : {opt_stats._strategy.rsi_window}")
print(f"  sl_multiplier : {opt_stats._strategy.sl_multiplier}")
print(f"  ema_lookback  : {opt_stats._strategy.ema_lookback}")

Backtest.optimize:   0%|          | 0/90 [00:00<?, ?it/s]

Backtest.run:   0%|          | 0/1695 [00:00<?, ?bar/s]

  rsi_threshold : 42
  rsi_window    : 7
  sl_multiplier : 2.0
  ema_lookback  : 10
